# Script to process NEX-GDDP-CMIP6 model T2m data to determine summers, winters, autumns, and springs lengths per year

In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import fft

from cartopy import crs as ccrs, feature as cfeature
import netCDF4
from netCDF4 import Dataset
from datetime import datetime as dt
import matplotlib.ticker as mticker

import warnings
warnings.filterwarnings('ignore')

###################
# helper functions
###################
def SummerStatsFourier(temp_data, temp_data_next, threshold, show_plot, calc_tmax, calc_heat):
    '''

    Function that uses the input 75th percentile temperature in K to identify the summer from a time series of temperatures.
    Input data should already be set to 365 days.

    Args:
        temp_data: xarray of temperature data for a single grid cell for the year of interest
        temp_data_next: xarray of temperature data for a single grid cell for the following year (for SH summers)
        threshold: xarray for the 75th percentile threshold to use
        show_plot: output a plot of the time series with threshold, day of max, summer start/end shown
        calc_tmax: should the day of max temp during summer be calculated (slows processing)
        calc_heat: should the accumulated heat during summer be calculated (slows processing), includes mean temp and mean HS temp
        
    Returns:
        start day: DOY that daily mean t2m (from smoothed Fourier fit) first crossed above threshold (75th percentile)
        end day: DOY that daily mean t2m (from smoothed Fourier fit) first crossed below threshold (75th percentile) 
        day with max temp: during the summer period, the day that had the highest mean daily t2m 
        accumulated heat in summer: sum of degree days above threshold during summer, 0 if none
        summer length: number of days the smoothed fit to the t2m data is above threshold, 0 if none
        rmse: root mean squared error of Fourier fit to temperature time series
        mean temp: the mean temperature in K for the summer period, 0 if there was no time above threshold
        
    Example:
        start_day, last_day, day_of_max, acc_heat, summer_length, rmse, mean_temp = SummerStats(temps, temps_next, thresh, True, True, False)

    '''

    # default values
    first_day = 0 
    last_day = 0
    tmax_day = 0 
    summer_heat = 0 
    summer_length = 0
    rmse = 0.0
    mean_t = 0

    # split method based on NH or SH

    ######
    # Northern Hemisphere (uses current year only)
    if temp_data.lat.values >= 0:
        # extract DOY and T2m values for those days
        x1 = temp_data.time.dt.dayofyear.values
        y1 = temp_data.values

        # Fourier fit, up to 4th harmonic (mean + 3 more) as in Weller et al 2020
        ymean = np.mean(y1)
        y_norm = y1 - ymean
        # perform DFT of temp data
        coef = fft(y_norm)
        # low-pass filter: retain only the mean and the first three harmonics, set all other frequencies to zero
        # help from https://kls2177.github.io/Climate-and-Geophysical-Data-Analysis/chapters/Week6/filtering_in_freq2.html
        Z_lp = np.copy(coef)
        Z_lp[4:-3:] = 0.0  # we have to set both the positive and negative frequencies to zero
        # apply inverse fourier transform to convert back to time domain (just want the real part)
        y_norm_lp = np.real(np.fft.ifft(Z_lp))

        # add back in the original mean temp
        y_lp = y_norm_lp + ymean

        # RMSE for Fourier fit
        rmse = np.sqrt(np.sum((y1-y_lp)**2)/365.0)
    
        # create some x values for plotting  fit
        x_values = np.linspace(min(x1), max(x1), 365) 
    
        # use fit to get stats
        summer_length = np.count_nonzero(y_lp >= threshold.values)
        summer_days = np.where(y_lp >= threshold.values)[0]

        # deal with those that didn't have any days above threshold
        if summer_length == 0:
            first_day = 0
            last_day = 0
            if calc_tmax:
                # day with max temp isn't above threshold but still want the value so can be any day of the year
                tmax_day = int(temp_data.argmax() + 1)
            if calc_heat:
                # determine accumulated heat over summer as sum of degree days above threshold
                # in this case there are none
                summer_heat = 0
        else:
            # otherwise can get start/end days 
            first_day = int(x_values[min(summer_days)])
            
            # end day comes after first day so find next crossing of  fit
            # this is to be careful to not count until the end of a double crossing
            end_window = np.where((y_lp[first_day:365]) < threshold.values)

            # if it does cross back below threshold before the end of the "year"
            if np.count_nonzero(end_window):
                last_day = end_window[0][0] + first_day

            # otherwise it never crosses back below threshold for the "year"
            # so the last day is day 365
            else:
                last_day = 365
           
            # summer length is also the span between first and last + 1 to include the end day
            summer_length = last_day - first_day + 1    

            #print(summer_length)
            
            if calc_tmax:
                # day with max temp that is in summer period
                if first_day == last_day: #only one day of summer
                    tmax_day = first_day
                else:
                    # if summer_length is 0 then we already grab the day of max temp in the year
                    # but the day of max can also be the last day of the year so have to special case
                    if (temp_data[(first_day):(last_day+1)]).size == 0:
                        tmax_day = last_day
                    else:         
                        tmax_day = int((temp_data[(first_day):(last_day+1)].argmax() + 1) + first_day)        
            if calc_heat:
                # determine accumulated heat over summer as sum of degree days above threshold
                # also save mean temp
                nh_summ = temp_data[(first_day):(last_day+1)]

                #print(nh_summ)

                # total area relative to zero for regions above threshold line
                above_thresh = float(nh_summ.where(nh_summ.values >= threshold.values).sum())
                days_above_thresh = nh_summ.where(nh_summ.values >= threshold.values).count()
                
                # area below threshold line to be subtracted
                thresh_heat = float(nh_summ.where(nh_summ.values >= threshold.values).count()) * threshold.values[0]
                
                # calculate difference to get only the area between threshold and data
                summer_heat = above_thresh - thresh_heat

                # if there is a summer we can get the mean temp for the whole summer 
                if len(nh_summ) > 0:
                    mean_t = nh_summ.values.mean()
    
    ######
    # Southern Hemisphere (uses current year and following year)
    else: 
        # concat both years
        ds_two = xr.concat([temp_data, temp_data_next], dim="time")

        # get the current year for slicing
        input_year = temp_data.time.dt.year.values[0]

        # select July 1 - June 30 across the two years to do fit
        startdate = str(input_year)+'-07-01' # 182nd day of a normal year
        enddate = str(input_year+1)+'-06-30'
        sh_yr = ds_two.sel(time=slice(startdate,enddate))
        
        # DOY (relative to July 1) and T2m values for those days
        x1 = np.arange(1,len(sh_yr.time)+1) 
        y1 = sh_yr.values
        
        # Fourier fit, up to 4th harmonic (mean + 3 more) as in Weller et al 2020
        ymean = np.mean(y1)
        y_norm = y1 - ymean
        # perform DFT of temp data
        coef = fft(y_norm)
        # low-pass filter: retain only the mean and the first three harmonics, set all other frequencies to zero
        # help from https://kls2177.github.io/Climate-and-Geophysical-Data-Analysis/chapters/Week6/filtering_in_freq2.html
        Z_lp = np.copy(coef)
        Z_lp[4:-3:] = 0.0  # we have to set both the positive and negative frequencies to zero
        # apply inverse fourier transform to convert back to time domain (just want the real part)
        y_norm_lp = np.real(np.fft.ifft(Z_lp))

        # add back in the original mean temp
        y_lp = y_norm_lp + ymean

        # RMSE for Fourier fit
        rmse = np.sqrt(np.sum((y1-y_lp)**2)/365.0)
        
        # create some x values for plotting  fit
        x_values = np.linspace(min(x1), max(x1), 365) 
        summer_length = np.count_nonzero(y_lp >= threshold.values)
        summer_days = np.where(y_lp >= threshold.values)[0]

        # check if summer length is 0
        if summer_length == 0:
            first_day = 0
            last_day = 0
            if calc_tmax:
                # day with max temp isn't above threshold but still want the value so can be any day of the year
                tmax_day = int(sh_yr.argmax() + 1)
            if calc_heat:
                # determine accumulated heat over summer as sum of degree days above threshold
                # in this case there are none
                summer_heat = 0
        else:
            # otherwise can get start/end days but note these are relative to July 1
            first_day = int(x_values[min(summer_days)])

            # end day comes after first day so find next crossing of  fit
            # this is to be careful to not count until the end of a double crossing
            end_window = np.where((y_lp[first_day:365]) < threshold.values)

            # if it does cross back below threshold before the end of the "year"
            if np.count_nonzero(end_window):
                last_day = end_window[0][0] + first_day

            # otherwise it never crosses back below threshold for the "year"
            # so the last day is day 365
            else:
                last_day = 365
           
            # summer length is the span between first and last + 1 to include the end day
            summer_length = last_day - first_day + 1           
           
            if calc_tmax:
                # day with max temp that is in summer period
                if first_day == last_day: #only one day of summer
                    tmax_day = first_day
                else:
                    # if summer_length is 0 then we already grab the day of max temp in the year
                    # but the day of max can also be the last day of the year so have to special case
                    if (sh_yr[(first_day):(last_day+1)]).size == 0:
                        tmax_day = last_day
                    else:         
                        tmax_day = int((sh_yr[(first_day):(last_day+1)].argmax() + 1) + first_day)  
            if calc_heat:
                # determine accumulated heat over summer as sum of degree days above threshold
                # also save the mean temp of summer
                sh_summ = sh_yr[(first_day):(last_day+1)]

                # total area relative to zero for regions above threshold line
                above_thresh = float(sh_summ.where(sh_summ.values >= threshold.values).sum())
                days_above_thresh = sh_summ.where(sh_summ.values >= threshold.values).count()
                
                # area below threshold line to be subtracted
                thresh_heat = float(sh_summ.where(sh_summ.values >= threshold.values).count()) * threshold.values[0]
                
                # calculate difference to get only the area between threshold and data
                summer_heat = above_thresh - thresh_heat

                # if there is a summer we can get the mean temp for the whole summer
                if len(sh_summ) > 0:
                    mean_t = sh_summ.values.mean()

                
    # make and save plot if desired
    ##
    ##
    if show_plot:
        cell_lat = temp_data.lat.values
        cell_lon = temp_data.lon.values
        yr = temp_data.time.dt.year.values[0]

        plt.figure(figsize=(16,6))
        plt.plot(x1,y1, label='daily mean T2m')
        plt.plot(x_values, y_lp, label='Fourier fit 1-4th Harmonic', color='red')
        plt.axhline(threshold.values[0], color="black", linewidth=0.8, linestyle="dashed") 
        plt.axvline(first_day, color="black", linewidth=0.8, linestyle="dashed")
        plt.axvline(last_day, color="black", linewidth=0.8, linestyle="dashed")
        plt.annotate('start', xy=(first_day-5,np.max(y_lp)+2), fontsize=12)
        plt.annotate('end', xy=(last_day-5,np.max(y_lp)+2), fontsize=12)
        plt.annotate('summer = '+str(summer_length)+' days', xy=(first_day+5,np.mean(y_lp)), fontsize=18)
        plt.annotate('75th percentile = '+str(np.round(threshold.values[0],1))+'K', xy=(0,threshold.values[0]+1), fontsize=18)
        plt.annotate('RMSE = '+str(np.round(rmse,2)), xy=(0, np.min(y1)), fontsize=18)
        plt.annotate('Mean = '+str(np.round(mean_t,2))+'K', xy=(last_day-5, mean_t-1), fontsize=12)

        if calc_tmax:
            # add annotation for tmax_day
            plt.annotate('Tmax day '+str(tmax_day), xy=(tmax_day,np.max(y1)), fontsize=12)

        if cell_lat < 0:
            # SH days are relative to July 1
            plt.xlabel('DOY (' + str(yr) + ') relative to July 1')
        else:
            plt.xlabel('DOY (' + str(yr) + ')')
        plt.ylabel('Mean Daily Temp (K)')
        plt.title(str(yr) + " Summer for Lat,Lon = [" +str(cell_lat)+","+str(cell_lon)+ "] Baseline Years = (1961-1990)")
        plt.legend()
        #plt.savefig("fourierfit_stats_"+str(cell_lat)+"_"+str(cell_lon)+"_"+str(yr)+".png")
        plt.show()
    
    # return
    return int(first_day), int(last_day), int(tmax_day), float(summer_heat), int(summer_length), float(rmse), float(mean_t)


# moving average for smoothing when winters have length zero
# help from: https://stackoverflow.com/questions/13728392/moving-average-or-running-mean/27681394#27681394
# using cumsum solution to optimize for speed
# here N is the number of days to include in the window
def RunningMean(x, N):
    cumsum = np.cumsum(np.insert(x, 0, 0)) 
    return (cumsum[N:] - cumsum[:-N]) / float(N)

def WinterStatsFourier(temp_data, temp_data_next, threshold, show_plot, calc_tmax, calc_cold):
    '''

    Function that uses the input 25th percentile temperature in K to identify the winter from a time series of temperatures.
    Input data should already be set to 365 days.

    Args:
        temp_data: xarray of temperature data for a single grid cell for the year of interest
        temp_data_next: xarray of temperature data for a single grid cell for the following year (for NH winters)
        threshold: xarray for the 25th percentile threshold to use
        show_plot: output plot of the time series with threshold, day of max, winter start/end shown
        calc_tmax: should the day of max temp during winter be calculated (slows processing)
        calc_cold: should the accumulated cold during winter be calculated (slows processing), includes mean temp
        
    Returns:
        start day: DOY that daily mean t2m (from smoothed Fourier fit) first crossed below threshold (25th percentile)
        end day: DOY that daily mean t2m (from smoothed Fourier fit) first crossed above threshold (25th percentile) 
        day with max temp: during the winter period, the day that had the highest mean daily t2m 
        accumulated cold in winter: sum of degree days below threshold during winter, 0 if none
        winter length: number of days the smoothed fit to the t2m data is below threshold, 0 if none
        rmse: root mean squared error of Fourier fit to temperature time series
        mean temp: the mean temperature in K for the winter period, 0 if there was no time below threshold
        
    Example:
        start_day, last_day, day_of_max, acc_cold, winter_length, rmse, mean_temp = WinterStats(temps, temps_next, thresh, True, False, True)

    '''

    # default values
    first_day = 0 
    last_day = 0
    tmax_day = 0 
    winter_cold = 0 
    winter_length = 0
    rmse = 0.0
    mean_t = 0

    # smoothing window for winter_length==0
    N = 91

    # split method based on NH or SH

    ######
    # Southern Hemisphere (uses current year only)
    if temp_data.lat.values < 0:
        # extract DOY and T2m values for those days
        x1 = temp_data.time.dt.dayofyear.values
        y1 = temp_data.values

        # Fourier fit, up to 4th harmonic (mean + 3 more) as in Weller et al 2020 and Scott et al 2026
        ymean = np.mean(y1)
        y_norm = y1 - ymean
        # perform DFT of temp data
        coef = fft(y_norm)
        # low-pass filter: retain only the mean and the first three harmonics, set all other frequencies to zero
        # help from https://kls2177.github.io/Climate-and-Geophysical-Data-Analysis/chapters/Week6/filtering_in_freq2.html
        Z_lp = np.copy(coef)
        Z_lp[4:-3:] = 0.0  # we have to set both the positive and negative frequencies to zero
        # apply inverse fourier transform to convert back to time domain (just want the real part)
        y_norm_lp = np.real(np.fft.ifft(Z_lp))

        # add back in the original mean temp
        y_lp = y_norm_lp + ymean

        # RMSE for Fourier fit
        rmse = np.sqrt(np.sum((y1-y_lp)**2)/365.0)
    
        # create some x values for plotting  fit
        x_values = np.linspace(min(x1), max(x1), 365) 
    
        # use fit to get stats
        # in Scott et al 2026 summer was >= threshold so using same definition for (winter <= threshold)
        winter_length = np.count_nonzero(y_lp <= threshold.values)
        winter_days = np.where(y_lp <= threshold.values)[0]

        # deal with those that didn't have any days below threshold
        # update: now smoothing and finding the lowest point in the smoothed series to be the occurrence of zero-length winter
        # this provides a dividing point between Autumn and Spring (all that is not Summer is then either Autumn or Spring)
        if winter_length == 0:
            # use RollingMean to smooth out any ripples in the Fourier fit
            # and identify the minimum to use for splitting Autumn from Spring
            # initial testing over 15-45 day windows confirm minimal impact of window size on min value

            y_smoothed = RunningMean(y_lp, N)
            # have to add half the window to match lowest point in smoothed fit
            first_day = int(y_smoothed.argmin() + 1 + int(N/2))
            last_day = first_day
            # first_day = 0
            # last_day = 0
            if calc_tmax:
                # day with max temp isn't below threshold but still want the value so can be any day of the year
                tmax_day = int(temp_data.argmax() + 1)
            if calc_cold:
                # determine accumulated cold over winter as sum of degree days below threshold
                # in this case there are none
                winter_cold = 0
        else:
            # otherwise can get start/end days 
            first_day = int(x_values[min(winter_days)])
            #OLD:# last_day = int(x_values[max(summer_days)])

            # end day comes after first day so find next crossing of  fit
            # this is to be careful to not count until the end of a double crossing
            end_window = np.where((y_lp[first_day:365]) > threshold.values)

            # if it does cross back above threshold before the end of the "year"
            if np.count_nonzero(end_window):
                last_day = end_window[0][0] + first_day

            # otherwise it never crosses back above threshold for the "year"
            # so the last day is day 365
            else:
                last_day = 365
           
            # winter length is the span between first and last + 1 to include the end day
            winter_length = last_day - first_day + 1    
            
            if calc_tmax:
                # day with max temp that is in winter period
                if first_day == last_day: #only one day of winter
                    tmax_day = first_day
                else:
                    # if winter_length is 0 then we already grab the day of max temp in the year
                    # but the day of max can also be the last day of the year so have to special case
                    if (temp_data[(first_day):(last_day+1)]).size == 0:
                        tmax_day = last_day
                    else:         
                        tmax_day = int((temp_data[(first_day):(last_day+1)].argmax() + 1) + first_day)        
            if calc_cold:
                # determine accumulated cold over winter as sum of degree (days) below threshold to daily mean temp
                # also save mean temp
                sh_winter = temp_data[(first_day):(last_day+1)]

                # area above zero for days below threshold line to be subtracted
                below_thresh_area = float(sh_winter.where(sh_winter.values <= threshold.values).sum())

                # number of days where mean temp was below threshold
                days_below_thresh = sh_winter.where(sh_winter.values <= threshold.values).count()
                
                # calculate difference to get only the area between threshold and data (daily mean temp on winter days)
                winter_cold = (threshold.values[0] * days_below_thresh) - below_thresh_area

                # if there is a winter we can get the mean temp for the whole winter
                if len(sh_winter) > 0:
                    mean_t = sh_winter.values.mean()

                
    ######
    # Northern Hemisphere (uses current year and following year)
    else: 
        # concat both years
        ds_two = xr.concat([temp_data, temp_data_next], dim="time")

        # get the current year for slicing
        input_year = temp_data.time.dt.year.values[0]

        # select July 1 - June 30 across the two years to do fit
        startdate = str(input_year)+'-07-01' # 182nd day of a normal year
        enddate = str(input_year+1)+'-06-30'
        nh_yr = ds_two.sel(time=slice(startdate,enddate))
        
        # DOY (relative to July 1) and T2m values for those days
        x1 = np.arange(1,len(nh_yr.time)+1) 
        y1 = nh_yr.values
        
        # Fourier fit, up to 4th harmonic (mean + 3 more) as in Weller et al 2020 and Scott et al 2026
        ymean = np.mean(y1)
        y_norm = y1 - ymean
        # perform DFT of temp data
        coef = fft(y_norm)
        # low-pass filter: retain only the mean and the first three harmonics, set all other frequencies to zero
        # help from https://kls2177.github.io/Climate-and-Geophysical-Data-Analysis/chapters/Week6/filtering_in_freq2.html
        Z_lp = np.copy(coef)
        Z_lp[4:-3:] = 0.0  # we have to set both the positive and negative frequencies to zero
        # apply inverse fourier transform to convert back to time domain (just want the real part)
        y_norm_lp = np.real(np.fft.ifft(Z_lp))

        # add back in the original mean temp
        y_lp = y_norm_lp + ymean

        # RMSE for Fourier fit
        rmse = np.sqrt(np.sum((y1-y_lp)**2)/365.0)
        
        # create some x values for plotting  fit
        x_values = np.linspace(min(x1), max(x1), 365) 
        winter_length = np.count_nonzero(y_lp <= threshold.values)
        winter_days = np.where(y_lp <=  threshold.values)[0]

        # check if winter length is 0
        if winter_length == 0:
            # use RollingMean to smooth out any ripples in the Fourier fit
            # and identify the minimum to use for splitting Autumn from Spring
            # initial testing over 15-45 day windows confirm minimal impact of window size on min value
            y_smoothed = RunningMean(y_lp, N)
            # have to add half the window to match lowest point in smoothed fit
            first_day = int(y_smoothed.argmin() + 1 + int(N/2))
            last_day = first_day
            if calc_tmax:
                # day with max temp isn't above threshold but still want the value so can be any day of the year
                tmax_day = int(nh_yr.argmax() + 1)
            if calc_cold:
                # determine accumulated heat over winter as sum of degree days below threshold
                # in this case there are none
                winter_cold = 0
        else:
            # otherwise can get start/end days but note these are relative to July 1
            first_day = int(x_values[min(winter_days)])

            # end day comes after first day so find next crossing of  fit
            # this is to be careful to not count until the end of a double crossing
            end_window = np.where((y_lp[first_day:365]) > threshold.values)

            # if it does cross back above threshold before the end of the "year"
            if np.count_nonzero(end_window):
                last_day = end_window[0][0] + first_day

            # otherwise it never crosses back above threshold for the "year"
            # so the last day is day 365
            else:
                last_day = 365
           
            # winter length is the span between first and last + 1 to include the end day
            winter_length = last_day - first_day + 1           
     
            if calc_tmax:
                # day with max temp that is in winter period
                if first_day == last_day: #only one day of winter
                    tmax_day = first_day
                else:
                    # if winter_length is 0 then we already grab the day of max temp in the year
                    # but the day of max can also be the last day of the year so have to special case
                    if (nh_yr[(first_day):(last_day+1)]).size == 0:
                        tmax_day = last_day
                    else:         
                        tmax_day = int((nh_yr[(first_day):(last_day+1)].argmax() + 1) + first_day)  
            if calc_cold:
                # determine accumulated cold over winter as sum of degree (days) below threshold to daily mean temp
                # also save the mean temp of winter
                nh_winter = nh_yr[(first_day):(last_day+1)]

                # area above zero for days below threshold line to be subtracted
                below_thresh_area = float(nh_winter.where(nh_winter.values <= threshold.values).sum())

                # number of days where mean temp was below threshold
                days_below_thresh = nh_winter.where(nh_winter.values <= threshold.values).count()
                
                # calculate difference to get only the area between threshold and data (daily mean temp on winter days)
                winter_cold = (threshold.values[0] * days_below_thresh) - below_thresh_area
    
                # if there is a winter we can get the mean temp for the whole winter
                if len(nh_winter) > 0:
                    mean_t = nh_winter.values.mean()
                
                
    # make and save plot if desired
    ##
    ##
    if show_plot:
        cell_lat = temp_data.lat.values
        cell_lon = temp_data.lon.values
        yr = temp_data.time.dt.year.values[0]

        plt.figure(figsize=(16,6))
        plt.plot(x1,y1, label='daily mean T2m')
        plt.plot(x_values, y_lp, label='Fourier fit 1-4th Harmonic', color='red')
        if winter_length == 0:
            #plt.plot(np.arange(1, len(y_smoothed)+1), y_smoothed, label='29d smooth', linestyle='dashed')
            xs = np.linspace(1, 365, len(y_smoothed))
            plt.plot(xs, y_smoothed, label='Xd smooth', linestyle='dashed')
        plt.axhline(threshold.values[0], color="black", linewidth=0.8, linestyle="dashed") 
        plt.axvline(first_day, color="black", linewidth=0.8, linestyle="dashed")
        plt.axvline(last_day, color="black", linewidth=0.8, linestyle="dashed")
        plt.annotate('start', xy=(first_day-5,np.max(y_lp)+2), fontsize=12)
        plt.annotate('end', xy=(last_day-5,np.max(y_lp)+2), fontsize=12)
        plt.annotate('winter = '+str(winter_length)+' days', xy=(first_day+5,np.mean(y_lp)), fontsize=18)
        plt.annotate('25th percentile = '+str(np.round(threshold.values[0],1))+'K', xy=(0,threshold.values[0]+1), fontsize=18)
        plt.annotate('RMSE = '+str(np.round(rmse,2)), xy=(0, np.min(y1)), fontsize=18)
        plt.annotate('Mean = '+str(np.round(mean_t,2))+'K', xy=(last_day-5, mean_t-1), fontsize=12)

        if calc_tmax:
            # add annotation for tmax_day
            plt.annotate('Tmax day '+str(tmax_day), xy=(tmax_day,np.max(y1)), fontsize=12)

        if cell_lat > 0:
            # NH days are relative to July 1
            plt.xlabel('DOY (' + str(yr) + ') relative to July 1')
        else:
            plt.xlabel('DOY (' + str(yr) + ')')
        plt.ylabel('Mean Daily Temp (K)')
        plt.title(str(yr) + " Winter for Lat,Lon = [" +str(cell_lat)+","+str(cell_lon)+ "] Baseline Years = (1961-1990)")
        plt.legend()
        #plt.savefig("fourierfit_stats_"+str(cell_lat)+"_"+str(cell_lon)+"_"+str(yr)+"_winter.png")
        plt.show()
    
    # return
    return int(first_day), int(last_day), int(tmax_day), float(winter_cold), int(winter_length), float(rmse), float(mean_t)




## Turns out some models include leap days

In [3]:
# a function that takes in a dataset of global t2m values and if it is a leap year averages
# the temps on Feb 28 & Feb 29, then drops Feb 29 to ensure the year has 365 days
def HandleLeapYears(input_ds):
    '''

    Function that takes the mean of Feb 28 & Feb 29 (if it exists) and returns an xarray Dataset that has 365 days
    where Feb 28 will now have a tas value that is the mean of Feb 28 & Feb 29

    Args:
        input_ds: the data set containing a single year of data that may need adjusting for leap year
        
    Returns:
        output_ds
        
    Example:
        no_leap_ds = HandleLeapYears(input_ds)

    '''
    

    # check if a leap year
    if(input_ds.time.dt.is_leap_year[0].values):
        
        # make deep copy
        output_ds = input_ds.copy(deep=True)
        
        # get mean of Feb 28 (time index 58) & Feb 29 (time index 59) for each grid cell and overwrite Feb 28 with it
        mean_dat = output_ds.tas[58:60].mean(dim='time', skipna=True, keep_attrs=True)
        output_ds.tas[58] = mean_dat
        
        # drop Feb 29
        output_ds = output_ds.convert_calendar('noleap', use_cftime=None)

        # ensure time is datetime64
        output_ds["time"] = output_ds.time.dt.strftime('%Y-%m-%d')
        output_ds['time'] = pd.to_datetime(output_ds['time'])

        # return the new dataset 
        return output_ds
    else:
        return input_ds
    

## Do summers

In [ ]:
%%time

# 15.5 min per year
# roughly 14 hrs per 54 years

# comment out as needed
models = [#'ACCESS-ESM1-5',
           #'BCC-CSM2-MR',
           #'CMCC-ESM2',
           #'FGOALS-g3',
           #'GFDL-ESM4',
           'GISS-E2-1-G',
           #'NorESM2-LM'
         ]#,'','','','','','','']

# run through the list of models
for md in models:

    start = dt.now()

    # summer thresh
    file_path1 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/computed_1961-1990-full_75th.nc'
    c_75 = xr.open_dataarray(file_path1)

    # grid parameters for looping
    num_lat = c_75.lat.shape[0]
    num_lon = c_75.lon.shape[0]
    loop_lats = c_75.lat.values[:num_lat]
    loop_lons = c_75.lon.values[:num_lon]

    ######
    # Historical period
    process_yrs = np.arange(1961, 2015,1) 
    
    # loop thru the years in the defined array above
    for input_year in process_yrs:
    
        start_yr = dt.now()

        # don't need to load in current year if we've already loaded it
        if input_year == 1961:
            # read in current year and next year
            # load into memory to speed up the summer stats processing and handle leap years to be 365 days
            file_path1 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_historical_'+str(input_year)+'.nc'
            dsnow = xr.open_mfdataset(file_path1)
            dsnow = HandleLeapYears(dsnow)
            dsnow.load()
        else:
            dsnow = dsnext

        # Note: to process 2014 we need 2015 for SH summer so using ssp245 2015 for that one added year\
        if input_year < 2014:
            file_path2 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_historical_'+str(input_year+1)+'.nc'
        else:
            file_path2 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_ssp245_2015.nc'
        dsnext = xr.open_mfdataset(file_path2)
        dsnext = HandleLeapYears(dsnext)
        dsnext.load()
    
        # empty data arrays for summer stats
        summerlength = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                    coords=[loop_lats, loop_lons], 
                                    dims=['lat','lon'])
        summerstart = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                   coords=[loop_lats, loop_lons], 
                                   dims=['lat','lon'])
        summerend = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        summermax = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        summerheat = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        summerrmse = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        summermean = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        
        # add attributes
        summerstart.attrs["long_name"] = "Start day of summer for the year (DOY in [1,365])"
        summerstart.attrs["units"] = "Day of year"
        
        summerend.attrs["long_name"] = "Last day of summer for the year (DOY in [1,365])"
        summerend.attrs["units"] = "Day of year"
        
        summerlength.attrs["long_name"] = "Duration of summer for the year"
        summerlength.attrs["units"] = "Days"
        
        summermax.attrs["long_name"] = "Day of max mean temp during summer for the year"
        summermax.attrs["units"] = "Day of year"
        
        summerheat.attrs["long_name"] = "Accumulated heat during summer for the year"
        summerheat.attrs["units"] = "Degree-Days"
        
        summerrmse.attrs["long_name"] = "Root mean squared error of Fourier fit"
        summerrmse.attrs["units"] = "Degrees [K]"
        
        summermean.attrs["long_name"] = "Mean temperature during the summer period"
        summermean.attrs["units"] = "Degrees [K]"
    
        # loop through all of the lat,lon of interest
        for input_lat in loop_lats:
            for input_lon in loop_lons:
                # get threshold from c_75 
                thresh = c_75.sel(lat=input_lat,lon=input_lon)

                # if thresh is nan this cell is not on land and/or outside lat range of data so skip it and set all metrics to nan
                if(np.isnan(thresh[0])):
                    start_day, last_day, day_of_max, acc_heat, summer_length, rmse, mean_temp = np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan 
                
                else:
                    # all the temps for that grid cell over time for this year and next (for SH)
                    curr = dsnow.tas.sel(lat=input_lat,lon=input_lon) 
                    nxt = dsnext.tas.sel(lat=input_lat, lon=input_lon)
            
                    # build summer stats for the grid cell and produce the plot if desired (currently False)
                    start_day, last_day, day_of_max, acc_heat, summer_length, rmse, mean_temp = SummerStatsFourier(curr, nxt, thresh, False, True, True)
                
                # xarray requires strange indexing in order to replace values by coordinate
                summerstart.loc[input_lat, input_lon] = start_day
                summerend.loc[input_lat, input_lon] = last_day
                summermax.loc[input_lat, input_lon] = day_of_max
                summerheat.loc[input_lat, input_lon] = acc_heat
                summerlength.loc[input_lat, input_lon] = summer_length
                summerrmse.loc[input_lat, input_lon] = rmse
                summermean.loc[input_lat, input_lon] = mean_temp
                
        # save it out
        sum_ds = xr.Dataset(data_vars={'SummerStart':summerstart,
                                   'SummerEnd':summerend,
                                   'SummerTmax':summermax,
                                   'SummerHeat':summerheat,
                                   'SummerLength':summerlength,
                                   'SummerRMSE':summerrmse,
                                    'SummerMeanT':summermean})

        output_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Summers/'+str(input_year)+'_'+md+'_historical_summer_stats.nc'
        sum_ds.to_netcdf(output_path)

        end_yr = dt.now()
        elapsed = (end_yr-start_yr).total_seconds()/60.0
        print("Time for "+str(input_year)+" :"+str(elapsed)+" min")


    ######
    # SSP2-4.5 Scenario
    # can only go up through 2099 since we use 2100 for the 2099 SH summer and don't have 2101 for the 2100 SH summer
    process_yrs = np.arange(2015, 2100,1)

    # loop thru the years in the defined array above
    for input_year in process_yrs:
    
        start_yr = dt.now()

        # read in current year and next year
        # don't need to load in current year if we've already loaded it as the next year
        if input_year == 2015:
            # load into memory to speed up the summer stats processing and handle leap years to be 365 days
            file_path1 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_ssp245_'+str(input_year)+'.nc'
            dsnow = xr.open_mfdataset(file_path1)
            dsnow = HandleLeapYears(dsnow)
            dsnow.load()
        else:
            dsnow = dsnext

        # Note: can't get SH summer for 2100 since we'd need 2101
        file_path2 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_ssp245_'+str(input_year+1)+'.nc'
        dsnext = xr.open_mfdataset(file_path2)
        dsnext = HandleLeapYears(dsnext)
        dsnext.load()
    
        # empty data arrays for summer stats
        summerlength = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                    coords=[loop_lats, loop_lons], 
                                    dims=['lat','lon'])
        summerstart = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                   coords=[loop_lats, loop_lons], 
                                   dims=['lat','lon'])
        summerend = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        summermax = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        summerheat = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        summerrmse = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        summermean = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        
        # add attributes
        summerstart.attrs["long_name"] = "Start day of summer for the year (DOY in [1,365])"
        summerstart.attrs["units"] = "Day of year"
        
        summerend.attrs["long_name"] = "Last day of summer for the year (DOY in [1,365])"
        summerend.attrs["units"] = "Day of year"
        
        summerlength.attrs["long_name"] = "Duration of summer for the year"
        summerlength.attrs["units"] = "Days"
        
        summermax.attrs["long_name"] = "Day of max mean temp during summer for the year"
        summermax.attrs["units"] = "Day of year"
        
        summerheat.attrs["long_name"] = "Accumulated heat during summer for the year"
        summerheat.attrs["units"] = "Degree-Days"
        
        summerrmse.attrs["long_name"] = "Root mean squared error of Fourier fit"
        summerrmse.attrs["units"] = "Degrees [K]"
        
        summermean.attrs["long_name"] = "Mean temperature during the summer period"
        summermean.attrs["units"] = "Degrees [K]"
    
        # loop through all of the lat,lon of interest
        for input_lat in loop_lats:
            for input_lon in loop_lons:
                # get threshold from c_75 
                thresh = c_75.sel(lat=input_lat,lon=input_lon)

                # if thresh is nan this cell is not on land and/or outside lat range of data so skip it and set all metrics to nan
                if(np.isnan(thresh[0])):
                    start_day, last_day, day_of_max, acc_heat, summer_length, rmse, mean_temp = np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan 
                
                else:
                    # all the temps for that grid cell over time for this year and next (for SH)
                    curr = dsnow.tas.sel(lat=input_lat,lon=input_lon) 
                    nxt = dsnext.tas.sel(lat=input_lat, lon=input_lon)
            
                    # build summer stats for the grid cell and produce the plot if desired (currently False)
                    start_day, last_day, day_of_max, acc_heat, summer_length, rmse, mean_temp = SummerStatsFourier(curr, nxt, thresh, False, True, True)
                
                # xarray requires strange indexing in order to replace values by coordinate
                summerstart.loc[input_lat, input_lon] = start_day
                summerend.loc[input_lat, input_lon] = last_day
                summermax.loc[input_lat, input_lon] = day_of_max
                summerheat.loc[input_lat, input_lon] = acc_heat
                summerlength.loc[input_lat, input_lon] = summer_length
                summerrmse.loc[input_lat, input_lon] = rmse
                summermean.loc[input_lat, input_lon] = mean_temp
                
        # save it out
        sum_ds = xr.Dataset(data_vars={'SummerStart':summerstart,
                                   'SummerEnd':summerend,
                                   'SummerTmax':summermax,
                                   'SummerHeat':summerheat,
                                   'SummerLength':summerlength,
                                   'SummerRMSE':summerrmse,
                                    'SummerMeanT':summermean})

        output_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Summers/'+str(input_year)+'_'+md+'_ssp245_summer_stats.nc'
        sum_ds.to_netcdf(output_path)

        end_yr = dt.now()
        elapsed = (end_yr-start_yr).total_seconds()/60.0
        print("Time for "+str(input_year)+" :"+str(elapsed)+" min")

    ######
    # SSP3-7.0

    ######
    # SSP1-2.6
    
    # total time for the Model being processed
    end = dt.now()
    elapsed = (end-start).total_seconds()/60.0
    print("Time for "+md+" :"+str(elapsed)+" min")

Time for 1961 :16.30797245 min
Time for 1962 :15.617685583333333 min
Time for 1963 :15.87827565 min
Time for 1964 :16.286592566666666 min
Time for 1965 :16.001258 min
Time for 1966 :16.027566999999998 min
Time for 1967 :15.972261933333334 min
Time for 1968 :16.144547550000002 min
Time for 1969 :15.942784816666666 min
Time for 1970 :16.087705483333334 min
Time for 1971 :16.220034733333332 min
Time for 1972 :16.340787 min
Time for 1973 :16.116791616666667 min
Time for 1974 :16.142206183333332 min
Time for 1975 :16.014619866666667 min
Time for 1976 :16.708347416666665 min
Time for 1977 :17.133440083333333 min
Time for 1978 :16.5370518 min
Time for 1979 :16.2635675 min
Time for 1980 :16.57096183333333 min
Time for 1981 :16.625292833333333 min


## Do winters

In [5]:
%%time

# 15.5 min per year
# roughly 14 hrs per 54 years

# comment out as needed
#models = ['ACCESS-ESM1-5',
           #'BCC-CSM2-MR',
           #'CMCC-ESM2',
           #'FGOALS-g3',
           #'GFDL-ESM4',
           #'GISS-E2-1-G',
           #'NorESM2-LM'
#         ]#,'','','','','','','']

# run through the list of models
for md in models:

    start = dt.now()

    # winter thresh
    file_path1 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/computed_1961-1990-full_25th.nc'
    c_25 = xr.open_dataarray(file_path1)

    # grid parameters for looping
    num_lat = c_25.lat.shape[0]
    num_lon = c_25.lon.shape[0]
    loop_lats = c_25.lat.values[:num_lat]
    loop_lons = c_25.lon.values[:num_lon]

    ######
    # Historical period
    process_yrs = np.arange(1961, 2015,1) 
    
    # loop thru the years in the defined array above
    for input_year in process_yrs:
    
        start_yr = dt.now()

        # don't need to load in current year if we've already loaded it
        if input_year == 1961:
            # read in current year and next year
            # load into memory to speed up the winter stats processing and handle leap years to be 365 days
            file_path1 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_historical_'+str(input_year)+'.nc'
            dsnow = xr.open_mfdataset(file_path1)
            dsnow = HandleLeapYears(dsnow)
            dsnow.load()
        else:
            dsnow = dsnext

        # Note: to process 2014 we need 2015 for NH Winter so using ssp245 2015 for that one added year\
        if input_year < 2014:
            file_path2 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_historical_'+str(input_year+1)+'.nc'
        else:
            file_path2 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_ssp245_2015.nc'
        dsnext = xr.open_mfdataset(file_path2)
        dsnext = HandleLeapYears(dsnext)
        dsnext.load()

        # empty data arrays for winter stats
        winterlength = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                    coords=[loop_lats, loop_lons], 
                                    dims=['lat','lon'])
        winterstart = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                   coords=[loop_lats, loop_lons], 
                                   dims=['lat','lon'])
        winterend = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        wintermax = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        wintercold = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        winterrmse = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        wintermean = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        
        # add attributes
        winterstart.attrs["long_name"] = "Start day of winter for the year (DOY in [1,365])"
        winterstart.attrs["units"] = "Day of year"
        
        winterend.attrs["long_name"] = "Last day of winter for the year (DOY in [1,365])"
        winterend.attrs["units"] = "Day of year"
        
        winterlength.attrs["long_name"] = "Duration of winter for the year"
        winterlength.attrs["units"] = "Days"
        
        wintermax.attrs["long_name"] = "Day of max mean temp during winter for the year"
        wintermax.attrs["units"] = "Day of year"
        
        wintercold.attrs["long_name"] = "Accumulated cold during winter for the year"
        wintercold.attrs["units"] = "Degree-Days"
        
        winterrmse.attrs["long_name"] = "Root mean squared error of Fourier fit"
        winterrmse.attrs["units"] = "Degrees [K]"
    
        wintermean.attrs["long_name"] = "Mean temperature during the winter period"
        wintermean.attrs["units"] = "Degrees [K]"
        
    
        # loop through all of the lat,lon of interest
        for input_lat in loop_lats:
            for input_lon in loop_lons:
                # get threshold from c_75 
                thresh = c_25.sel(lat=input_lat,lon=input_lon)
    
                # if thresh is nan this cell is not on land and/or outside lat range of data so skip it and set all metrics to nan
                if(np.isnan(thresh[0])):
                    start_day, last_day, day_of_max, acc_cold, winter_length, rmse, mean_temp = np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan 
                
                else:
                    # all the temps for that grid cell over time for this year and next (for NH)
                    curr = dsnow.tas.sel(lat=input_lat,lon=input_lon) 
                    nxt = dsnext.tas.sel(lat=input_lat, lon=input_lon)
        
                    # build winter stats for the grid cell and produce the plot if desired (currently False)
                    start_day, last_day, day_of_max, acc_cold, winter_length, rmse, mean_temp = WinterStatsFourier(curr, nxt, thresh, False, True, True)
                
                # xarray requires strange indexing in order to replace values by coordinate
                winterstart.loc[input_lat, input_lon] = start_day
                winterend.loc[input_lat, input_lon] = last_day
                wintermax.loc[input_lat, input_lon] = day_of_max
                wintercold.loc[input_lat, input_lon] = acc_cold
                winterlength.loc[input_lat, input_lon] = winter_length
                winterrmse.loc[input_lat, input_lon] = rmse
                wintermean.loc[input_lat, input_lon] = mean_temp
    
        # save it out
        sum_ds = xr.Dataset(data_vars={'WinterStart':winterstart,
                                   'WinterEnd':winterend,
                                   'WinterTmax':wintermax,
                                   'WinterCold':wintercold,
                                   'WinterLength':winterlength,
                                   'WinterRMSE':winterrmse,
                                    'WinterMeanT':wintermean})
        output_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Winters/'+str(input_year)+'_'+md+'_historical_winter_stats.nc'
        sum_ds.to_netcdf(output_path)
    
    
        end_yr = dt.now()
        elapsed = (end_yr-start_yr).total_seconds()/60.0
        print("Time for "+str(input_year)+" :"+str(elapsed)+" min")


    ######
    # SSP2-4.5 Scenario
    # can only go up through 2099 since we use 2100 for the 2099 NH winter and don't have 2101 for the 2100 NH winter
    process_yrs = np.arange(2015, 2100,1)

    # loop thru the years in the defined array above
    for input_year in process_yrs:
    
        start_yr = dt.now()

        # read in current year and next year
        # don't need to load in current year if we've already loaded it as the next year
        if input_year == 2015:
            # load into memory to speed up the summer stats processing and handle leap years to be 365 days
            file_path1 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_ssp245_'+str(input_year)+'.nc'
            dsnow = xr.open_mfdataset(file_path1)
            dsnow = HandleLeapYears(dsnow)
            dsnow.load()
        else:
            dsnow = dsnext

        # Note: can't get SH summer for 2100 since we'd need 2101
        file_path2 = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/tas_'+md+'_ssp245_'+str(input_year+1)+'.nc'
        dsnext = xr.open_mfdataset(file_path2)
        dsnext = HandleLeapYears(dsnext)
        dsnext.load()

        # empty data arrays for winter stats
        winterlength = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                    coords=[loop_lats, loop_lons], 
                                    dims=['lat','lon'])
        winterstart = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                   coords=[loop_lats, loop_lons], 
                                   dims=['lat','lon'])
        winterend = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        wintermax = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        wintercold = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        winterrmse = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        wintermean = xr.DataArray(np.zeros((num_lat,num_lon)), 
                                 coords=[loop_lats, loop_lons], 
                                 dims=['lat','lon'])
        
        # add attributes
        winterstart.attrs["long_name"] = "Start day of winter for the year (DOY in [1,365])"
        winterstart.attrs["units"] = "Day of year"
        
        winterend.attrs["long_name"] = "Last day of winter for the year (DOY in [1,365])"
        winterend.attrs["units"] = "Day of year"
        
        winterlength.attrs["long_name"] = "Duration of winter for the year"
        winterlength.attrs["units"] = "Days"
        
        wintermax.attrs["long_name"] = "Day of max mean temp during winter for the year"
        wintermax.attrs["units"] = "Day of year"
        
        wintercold.attrs["long_name"] = "Accumulated cold during winter for the year"
        wintercold.attrs["units"] = "Degree-Days"
        
        winterrmse.attrs["long_name"] = "Root mean squared error of Fourier fit"
        winterrmse.attrs["units"] = "Degrees [K]"
    
        wintermean.attrs["long_name"] = "Mean temperature during the winter period"
        wintermean.attrs["units"] = "Degrees [K]"
        
    
        # loop through all of the lat,lon of interest
        for input_lat in loop_lats:
            for input_lon in loop_lons:
                # get threshold from c_75 
                thresh = c_25.sel(lat=input_lat,lon=input_lon)
    
                # if thresh is nan this cell is not on land and/or outside lat range of data so skip it and set all metrics to nan
                if(np.isnan(thresh[0])):
                    start_day, last_day, day_of_max, acc_cold, winter_length, rmse, mean_temp = np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan 
                
                else:
                    # all the temps for that grid cell over time for this year and next (for NH)
                    curr = dsnow.tas.sel(lat=input_lat,lon=input_lon) 
                    nxt = dsnext.tas.sel(lat=input_lat, lon=input_lon)
        
                    # build winter stats for the grid cell and produce the plot if desired (currently False)
                    start_day, last_day, day_of_max, acc_cold, winter_length, rmse, mean_temp = WinterStatsFourier(curr, nxt, thresh, False, True, True)
                
                # xarray requires strange indexing in order to replace values by coordinate
                winterstart.loc[input_lat, input_lon] = start_day
                winterend.loc[input_lat, input_lon] = last_day
                wintermax.loc[input_lat, input_lon] = day_of_max
                wintercold.loc[input_lat, input_lon] = acc_cold
                winterlength.loc[input_lat, input_lon] = winter_length
                winterrmse.loc[input_lat, input_lon] = rmse
                wintermean.loc[input_lat, input_lon] = mean_temp
    
        # save it out
        sum_ds = xr.Dataset(data_vars={'WinterStart':winterstart,
                                   'WinterEnd':winterend,
                                   'WinterTmax':wintermax,
                                   'WinterCold':wintercold,
                                   'WinterLength':winterlength,
                                   'WinterRMSE':winterrmse,
                                    'WinterMeanT':wintermean})
        output_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Winters/'+str(input_year)+'_'+md+'_ssp245_winter_stats.nc'
        sum_ds.to_netcdf(output_path)
    
    
        end_yr = dt.now()
        elapsed = (end_yr-start_yr).total_seconds()/60.0
        print("Time for "+str(input_year)+" :"+str(elapsed)+" min")


    ######
    # SSP3-7.0

    ######
    # SSP1-2.6
    
    # total time for the Model being processed
    end = dt.now()
    elapsed = (end-start).total_seconds()/60.0
    print("Time for "+md+" :"+str(elapsed)+" min")

Time for 1961 :17.911038883333333 min
Time for 1962 :18.511244266666665 min
Time for 1963 :18.6421144 min
Time for 1964 :18.31570053333333 min
Time for 1965 :18.680548916666666 min
Time for 1966 :18.621383783333332 min
Time for 1967 :18.514137249999997 min
Time for 1968 :18.498256533333333 min
Time for 1969 :18.87618386666667 min
Time for 1970 :126.58255915 min
Time for 1971 :54.44144861666667 min
Time for 1972 :18.659748516666664 min
Time for 1973 :18.253033683333335 min
Time for 1974 :18.011432916666667 min
Time for 1975 :17.984013249999997 min
Time for 1976 :17.859234083333334 min
Time for 1977 :17.576079783333334 min
Time for 1978 :17.78513171666667 min
Time for 1979 :17.66556458333333 min
Time for 1980 :18.1310458 min
Time for 1981 :18.103896766666665 min
Time for 1982 :17.74371145 min
Time for 1983 :17.946054916666665 min
Time for 1984 :17.607145816666666 min
Time for 1985 :18.157151916666667 min
Time for 1986 :18.202251966666665 min
Time for 1987 :18.138208966666664 min
Time for

In [9]:
sum_ds

<xarray.Dataset> Size: 58MB
Dimensions:       (lat: 721, lon: 1440)
Coordinates:
  * lat           (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon           (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.2 179.5 179.8
Data variables:
    WinterStart   (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan
    WinterEnd     (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan
    WinterTmax    (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan
    WinterCold    (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan
    WinterLength  (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan
    WinterRMSE    (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan
    WinterMeanT   (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan

## Do Autumn and Spring

### test two years, Vancouver and Sydney

In [20]:
%%time

#md = 'ACCESS-ESM1-5'

##################
# Historical Summers: start with one year and add a coordinate for the year and model
year = 1961
input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Summers/'+str(year)+'_'+md+'_historical_summer_stats.nc'
ds_summ = xr.open_dataset(input_path)
ds_summ = ds_summ.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))

# loop thru remaining years and also add time coord
input_years = np.arange(1962,1964,1)

for year in input_years:
    input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Summers/'+str(year)+'_'+md+'_historical_summer_stats.nc'
    ds_i = xr.open_dataset(input_path)
    ds_i = ds_i.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
    # ds_i = ds_i.assign_coords(model=md)
    # ds_i = ds_i.expand_dims('model')
    ds_summ = ds_summ.merge(ds_i).drop_vars(["SummerTmax","SummerRMSE","SummerMeanT","SummerHeat"])

# Historical Winters
year = 1961
input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Winters/'+str(year)+'_'+md+'_historical_winter_stats.nc'
ds_wint = xr.open_dataset(input_path)
ds_wint = ds_wint.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
#ds_wint = ds_wint.expand_dims('model').assign_coords(model=md)

# loop thru remaining years and also add time coord
input_years = np.arange(1962,1964,1)

for year in input_years:
    input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Winters/'+str(year)+'_'+md+'_historical_winter_stats.nc'
    ds_i = xr.open_dataset(input_path)
    ds_i = ds_i.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
    #ds_i = ds_i.expand_dims('model').assign_coords(model=md)
    ds_wint = ds_wint.merge(ds_i).drop_vars(['WinterTmax','WinterCold','WinterRMSE','WinterMeanT'])

# merge all data
ds_seasons = ds_wint.merge(ds_summ)


# set spring and autumn lengths to zero initially but ensure they have coordinates
ds_seasons['SpringLength'] = ds_seasons.SummerLength - ds_seasons.SummerLength
ds_seasons['AutumnLength'] = ds_seasons.SummerLength - ds_seasons.SummerLength

# split by hemisphere since we process each differently
nh = ds_seasons.sel(lat=slice(0,90))
sh = ds_seasons.sel(lat=slice(-90,-0.25))

# SH springs are easy
sh['SpringLength'] = (sh.SummerStart + 183) - sh.WinterEnd

# SH autumns span two years
sh_autumn = sh.AutumnLength.copy(deep=True)
proc_yrs = np.arange(1962, 1964, 1)
for yr in proc_yrs:
    sh_autumn.loc[dict(lat=slice(-90,-0.25), lon=slice(-180,180), time=str(yr))] = sh.WinterStart.sel(time = str(yr)).values - (sh.SummerEnd.sel(time = str(yr - 1)) - 183).values

# NH autumns are easy
nh['AutumnLength'] = (nh.WinterStart + 183) - nh.SummerEnd

# NH Springs span two years
nh_spring = nh.SpringLength.copy(deep=True)
proc_yrs = np.arange(1962, 1964, 1)
for yr in proc_yrs:
    nh_spring.loc[dict(lat=slice(0,90), lon=slice(-180,180), time=str(yr))] = nh.SummerStart.sel(time = str(yr)).values - (nh.WinterEnd.sel(time = str(yr - 1)) - 183).values

# merge back together
nh['SpringLength'] = nh_spring
sh['AutumnLength'] = sh_autumn
combined = nh.merge(sh)
combined = combined.assign_coords(experiment='historical')
combined = combined.expand_dims('experiment')
combined = combined.assign_coords(model=md)
combined = combined.expand_dims('model')


combined


CPU times: user 388 ms, sys: 388 ms, total: 776 ms
Wall time: 866 ms


<xarray.Dataset> Size: 199MB
Dimensions:       (lat: 721, lon: 1440, time: 3, model: 1, experiment: 1)
Coordinates:
  * lat           (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon           (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.2 179.5 179.8
  * time          (time) datetime64[us] 24B 1961-01-01 1962-01-01 1963-01-01
  * model         (model) <U13 52B 'ACCESS-ESM1-5'
  * experiment    (experiment) <U10 40B 'historical'
Data variables:
    WinterStart   (model, experiment, time, lat, lon) float64 25MB nan ... nan
    WinterEnd     (model, experiment, time, lat, lon) float64 25MB nan ... nan
    WinterLength  (model, experiment, time, lat, lon) float64 25MB nan ... nan
    SummerStart   (model, experiment, time, lat, lon) float64 25MB nan ... nan
    SummerEnd     (model, experiment, time, lat, lon) float64 25MB nan ... nan
    SummerLength  (model, experiment, time, lat, lon) float64 25MB nan ... nan
    SpringLength  (model, experiment, time, lat, lon) float64 25MB nan ... nan
    AutumnLength  (model, experiment, time, lat, lon) float64 25MB nan ... nan

In [21]:
# Vancouver
combined.sel(lat=49.25,lon=-123)

<xarray.Dataset> Size: 324B
Dimensions:       (time: 3, model: 1, experiment: 1)
Coordinates:
  * time          (time) datetime64[us] 24B 1961-01-01 1962-01-01 1963-01-01
  * model         (model) <U13 52B 'ACCESS-ESM1-5'
  * experiment    (experiment) <U10 40B 'historical'
    lat           float64 8B 49.25
    lon           float64 8B -123.0
Data variables:
    WinterStart   (model, experiment, time) float64 24B 145.0 136.0 132.0
    WinterEnd     (model, experiment, time) float64 24B 290.0 238.0 282.0
    WinterLength  (model, experiment, time) float64 24B 146.0 103.0 151.0
    SummerStart   (model, experiment, time) float64 24B 160.0 171.0 147.0
    SummerEnd     (model, experiment, time) float64 24B 264.0 264.0 186.0
    SummerLength  (model, experiment, time) float64 24B 105.0 94.0 40.0
    SpringLength  (model, experiment, time) float64 24B 0.0 64.0 92.0
    AutumnLength  (model, experiment, time) float64 24B 64.0 55.0 129.0

In [22]:
# Sydney
combined.sel(lat=-33.75,lon=151.25)

<xarray.Dataset> Size: 324B
Dimensions:       (time: 3, model: 1, experiment: 1)
Coordinates:
  * time          (time) datetime64[us] 24B 1961-01-01 1962-01-01 1963-01-01
  * model         (model) <U13 52B 'ACCESS-ESM1-5'
  * experiment    (experiment) <U10 40B 'historical'
    lat           float64 8B -33.75
    lon           float64 8B 151.2
Data variables:
    WinterStart   (model, experiment, time) float64 24B 160.0 144.0 150.0
    WinterEnd     (model, experiment, time) float64 24B 260.0 246.0 242.0
    WinterLength  (model, experiment, time) float64 24B 101.0 103.0 93.0
    SummerStart   (model, experiment, time) float64 24B 156.0 147.0 172.0
    SummerEnd     (model, experiment, time) float64 24B 262.0 265.0 244.0
    SummerLength  (model, experiment, time) float64 24B 107.0 119.0 73.0
    SpringLength  (model, experiment, time) float64 24B 79.0 84.0 113.0
    AutumnLength  (model, experiment, time) float64 24B 0.0 65.0 68.0

In [26]:
%%time
# for each model we need to pull in the datasets for summer and winter and then compute autumn and spring lengths

# Roughly 8.5 min for one model and two experiments (historical, ssp245)

# comment out as needed
models = ['ACCESS-ESM1-5',
           #'BCC-CSM2-MR',
           #'CMCC-ESM2',
           #'FGOALS-g3',
           #'GFDL-ESM4',
           #'GISS-E2-1-G',
           #'NorESM2-LM'
         ]#,'','','','','','','']

# run through the list of models of interest
for md in models:

    start = dt.now()
    
    ######################################################################################################################
    # Historical Summers: start with one year and add a coordinate for the year and model
    year = 1961
    input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Summers/'+str(year)+'_'+md+'_historical_summer_stats.nc'
    ds_summ = xr.open_dataset(input_path)
    ds_summ = ds_summ.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
    
    # loop thru remaining years and also add time coord
    input_years = np.arange(1962,2015,1)
    
    for year in input_years:
        input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Summers/'+str(year)+'_'+md+'_historical_summer_stats.nc'
        ds_i = xr.open_dataset(input_path)
        ds_i = ds_i.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
        ds_summ = ds_summ.merge(ds_i).drop_vars(["SummerTmax","SummerRMSE","SummerMeanT","SummerHeat"])

    # Historical Winters
    year = 1961
    input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Winters/'+str(year)+'_'+md+'_historical_winter_stats.nc'
    ds_wint = xr.open_dataset(input_path)
    ds_wint = ds_wint.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))

    
    # loop thru remaining years and also add time coord
    input_years = np.arange(1962,2015,1)
    
    for year in input_years:
        input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Winters/'+str(year)+'_'+md+'_historical_winter_stats.nc'
        ds_i = xr.open_dataset(input_path)
        ds_i = ds_i.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
        ds_wint = ds_wint.merge(ds_i).drop_vars(['WinterTmax','WinterCold','WinterRMSE','WinterMeanT'])

    # merge all data
    ds_seasons = ds_wint.merge(ds_summ)
    
    # set spring and autumn lengths to zero initially but ensure they have coordinates
    ds_seasons['SpringLength'] = ds_seasons.SummerLength - ds_seasons.SummerLength
    ds_seasons['AutumnLength'] = ds_seasons.SummerLength - ds_seasons.SummerLength
    
    # split by hemisphere since we process each differently
    nh = ds_seasons.sel(lat=slice(0,90))
    sh = ds_seasons.sel(lat=slice(-90,-0.25))
    
    # SH springs are easy
    sh['SpringLength'] = (sh.SummerStart + 183) - sh.WinterEnd
    
    # SH autumns span two years
    sh_autumn = sh.AutumnLength.copy(deep=True)
    proc_yrs = np.arange(1962, 2015, 1)
    for yr in proc_yrs:
        sh_autumn.loc[dict(lat=slice(-90,-0.25), lon=slice(-180,180), time=str(yr))] = sh.WinterStart.sel(time = str(yr)).values - (sh.SummerEnd.sel(time = str(yr - 1)) - 183).values
    
    # NH autumns are easy
    nh['AutumnLength'] = (nh.WinterStart + 183) - nh.SummerEnd
    
    # NH Springs span two years
    nh_spring = nh.SpringLength.copy(deep=True)
    proc_yrs = np.arange(1962, 2015, 1)
    for yr in proc_yrs:
        nh_spring.loc[dict(lat=slice(0,90), lon=slice(-180,180), time=str(yr))] = nh.SummerStart.sel(time = str(yr)).values - (nh.WinterEnd.sel(time = str(yr - 1)) - 183).values
    
    # merge back together and assign model and experiment variables for ease of combining with other models
    nh['SpringLength'] = nh_spring
    sh['AutumnLength'] = sh_autumn
    combined = nh.merge(sh)
    combined = combined.assign_coords(experiment='historical')
    combined = combined.expand_dims('experiment')
    combined = combined.assign_coords(model=md)
    combined = combined.expand_dims('model')

    # write out historical season lengths dataset 
    output_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Seasons/1961-2014_'+md+'_historical_season_stats.nc'
    combined.to_netcdf(output_path)

    
    ########################################################################################################################
    # SSP2-4.5 Summers: start with one year and add a coordinate for the year and model
    year = 2015
    input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Summers/'+str(year)+'_'+md+'_ssp245_summer_stats.nc'
    ds_summ = xr.open_dataset(input_path)
    ds_summ = ds_summ.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
    
    # loop thru remaining years and also add time coord
    input_years = np.arange(2016,2100,1)
    
    for year in input_years:
        input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Summers/'+str(year)+'_'+md+'_ssp245_summer_stats.nc'
        ds_i = xr.open_dataset(input_path)
        ds_i = ds_i.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
        ds_summ = ds_summ.merge(ds_i).drop_vars(["SummerTmax","SummerRMSE","SummerMeanT","SummerHeat"])

    # SSP2-4.5 Winters
    year = 2015
    input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Winters/'+str(year)+'_'+md+'_ssp245_winter_stats.nc'
    ds_wint = xr.open_dataset(input_path)
    ds_wint = ds_wint.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
    
    # loop thru remaining years and also add time coord
    input_years = np.arange(2016,2100,1)
    
    for year in input_years:
        input_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Winters/'+str(year)+'_'+md+'_ssp245_winter_stats.nc'
        ds_i = xr.open_dataset(input_path)
        ds_i = ds_i.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
        ds_wint = ds_wint.merge(ds_i).drop_vars(['WinterTmax','WinterCold','WinterRMSE','WinterMeanT'])

    # merge all data
    ds_seasons = ds_wint.merge(ds_summ)

    
    # set spring and autumn lengths to zero initially but ensure they have coordinates
    ds_seasons['SpringLength'] = ds_seasons.SummerLength - ds_seasons.SummerLength
    ds_seasons['AutumnLength'] = ds_seasons.SummerLength - ds_seasons.SummerLength
    
    # split by hemisphere since we process each differently
    nh = ds_seasons.sel(lat=slice(0,90))
    sh = ds_seasons.sel(lat=slice(-90,-0.25))
    
    # SH springs are easy
    sh['SpringLength'] = (sh.SummerStart + 183) - sh.WinterEnd
    
    # SH autumns span two years
    sh_autumn = sh.AutumnLength.copy(deep=True)
    proc_yrs = np.arange(2016, 2100, 1)
    for yr in proc_yrs:
        sh_autumn.loc[dict(lat=slice(-90,-0.25), lon=slice(-180,180), time=str(yr))] = sh.WinterStart.sel(time = str(yr)).values - (sh.SummerEnd.sel(time = str(yr - 1)) - 183).values
    
    # NH autumns are easy
    nh['AutumnLength'] = (nh.WinterStart + 183) - nh.SummerEnd
    
    # NH Springs span two years
    nh_spring = nh.SpringLength.copy(deep=True)
    proc_yrs = np.arange(2016, 2100, 1)
    for yr in proc_yrs:
        nh_spring.loc[dict(lat=slice(0,90), lon=slice(-180,180), time=str(yr))] = nh.SummerStart.sel(time = str(yr)).values - (nh.WinterEnd.sel(time = str(yr - 1)) - 183).values
    
    # merge back together
    nh['SpringLength'] = nh_spring
    sh['AutumnLength'] = sh_autumn
    combined = nh.merge(sh)
    combined = combined.assign_coords(experiment='ssp245')
    combined = combined.expand_dims('experiment')
    combined = combined.assign_coords(model=md)
    combined = combined.expand_dims('model')

    # write out ssp245 season lengths dataset 
    output_path = '../../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/Seasons/2015-2099_'+md+'_ssp245_season_stats.nc'
    combined.to_netcdf(output_path)

    # total time for the Model being processed
    end = dt.now()
    elapsed = (end-start).total_seconds()/60.0
    print("Time for "+md+" :"+str(elapsed)+" min")

Time for ACCESS-ESM1-5 :8.552053616666665 min
CPU times: user 3min 10s, sys: 4min 30s, total: 7min 40s
Wall time: 8min 33s
